In [0]:
catalog_name = "catalog_name"
schema_name = "schema_name"
volume_name = "my_files"


spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

DataFrame[]

In [0]:
import pandas as pd
data = [
    ["3512", "Raoul", "USA", "Journalist"],
    ["3699", "Gonzo", "USA", "Doctor"],
    ["4242", "Slartibartfast", "MAG", "Architect"],
    ["3187", "Arthur", "GBR", "Writer"]
]

columns = ["ID", "FirstName", "Country", "Role"]
df = pd.DataFrame(data, columns=columns)
file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/employees.csv"
df.to_csv(file_path, index=False)

# df.display()

ID,FirstName,Country,Role
3512,Raoul,USA,Journalist
3699,Gonzo,USA,Doctor
4242,Slartibartfast,MAG,Architect
3187,Arthur,GBR,Writer


In [0]:
# set default catalog and schema
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")


DataFrame[]

In [0]:
# Set the default catalog and schema pyspark
spark.catalog.setCurrentCatalog(catalog_name)
spark.catalog.setCurrentDatabase(schema_name)

# display available tables in your schema
spark.catalog.listTables(schema_name)

[]

In [0]:
# look at files available within a specific volume

spark.sql(f"LIST '/Volumes/{catalog_name}/{schema_name}/{volume_name}/' ").display()

path,name,size,modification_time
/Volumes/catalog_name/schema_name/my_files/employees.csv,employees.csv,131,1774797937000


In [0]:
data = [
    ["0001", "Henry", "USA", "Janitor"],
    ["0011", "Jane", "USA", "Student"]
]
columns = ["ID", "FirstName", "Country", "Role"]

df = pd.DataFrame(data, columns=columns)
df.to_csv(f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/employees2.csv", index=False)

In [0]:
spark.sql(f"LIST '/Volumes/{catalog_name}/{schema_name}/{volume_name}/' ").display()

path,name,size,modification_time
/Volumes/catalog_name/schema_name/my_files/employees.csv,employees.csv,131,1774797937000
/Volumes/catalog_name/schema_name/my_files/employees2.csv,employees2.csv,71,1774799861000


In [0]:
%sql
-- Ingest Raw Data from CSV, create table first
DROP TABLE IF EXISTS current_employees_bronze;

CREATE TABLE current_employees_bronze (
  ID STRING,
  FirstName STRING,
  Country STRING,
  Role STRING
)


In [0]:
## Create the bronze raw ingestion table and include CSV file
## this will take any of the files in that volume
spark.sql(f'''
          COPY INTO current_employees_bronze
          FROM '/Volumes/{catalog_name}/{schema_name}/{volume_name}/'
          FILEFORMAT = CSV
          FORMAT_OPTIONS (
              'header' = 'true'
              )
          ''').display()

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
6,6,0


In [0]:
%sql
SELECT * 
FROM   current_employees_bronze

ID,FirstName,Country,Role
3512,Raoul,USA,Journalist
3699,Gonzo,USA,Doctor
4242,Slartibartfast,MAG,Architect
3187,Arthur,GBR,Writer
0001,Henry,USA,Janitor
0011,Jane,USA,Student


In [0]:
%sql
-- Cleaning steps into the silver layer
CREATE OR REPLACE TABLE current_employees_silver AS 
SELECT 
  ID,
  FirstName,
  upper(Role) as Role,
  current_timestamp() as Extract_Date_Time,
  date(current_timestamp) as CurrentDate
  FROM current_employees_bronze;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM current_employees_silver

ID,FirstName,Role,Extract_Date_Time,CurrentDate
3512,Raoul,JOURNALIST,2026-03-29T16:11:36.218Z,2026-03-29
3699,Gonzo,DOCTOR,2026-03-29T16:11:36.218Z,2026-03-29
4242,Slartibartfast,ARCHITECT,2026-03-29T16:11:36.218Z,2026-03-29
3187,Arthur,WRITER,2026-03-29T16:11:36.218Z,2026-03-29
0001,Henry,JANITOR,2026-03-29T16:11:36.218Z,2026-03-29
0011,Jane,STUDENT,2026-03-29T16:11:36.218Z,2026-03-29


In [0]:
%sql
CREATE TABLE IF NOT EXISTS total_roles_gold (
  Role STRING,
  TotalEmployees INT
)


In [0]:
%sql
INSERT OVERWRITE TABLE total_roles_gold
SELECT Role, count(*) as TotalEmployees
FROM current_employees_silver
GROUP BY Role

num_affected_rows,num_inserted_rows
6,6


In [0]:
%sql 
SELECT * 
FROM total_roles_gold

Role,TotalEmployees
DOCTOR,1
JOURNALIST,1
ARCHITECT,1
JANITOR,1
STUDENT,1
WRITER,1
